# 5 · Unit commitment: which power stations run?
### الطريق · Move — رحلة الكم من مصدر الطاقة إلى الاستدامة · Quantum Journey from Energy Source to Sustainability
*KACST Qiskit Fall Fest 2026, Riyadh · Team 5 · Prepared by Ahmed Elghadban, Quantumy*<br><sub>Reviewed by Ola Tarig</sub>

| | |
|---|---|
| Use case | Choose which generating units to switch on to meet demand at least cost |
| Algorithm | Grover adaptive search (Qiskit `GroverOptimizer`) |
| Toy size | 11 qubits, runs in under 3 min on a laptop |
| Day 1 | read sections 0–2 |
| Day 2 | sections 3–6, build, pitch |

In [ ]:
# Setup: run me first. On Colab this clones the repo and installs the pinned packages (about 1 minute).
import sys, os, subprocess
if "google.colab" in sys.modules and not os.path.exists("Quantum-Coding-Challenge"):
    subprocess.run(["git", "clone", "-q", "https://github.com/QTACInstitute/Quantum-Coding-Challenge.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "Quantum-Coding-Challenge/requirements.txt"], check=True)
ROOT = "Quantum-Coding-Challenge" if os.path.exists("Quantum-Coding-Challenge/qff") else ".."
sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, matplotlib.pyplot as plt
import qff
SEED = qff.SEED            # 2026
DATA = os.path.join(ROOT, "data")

## 0 · The journey so far — رحلة الكم
**«من المنبع إلى المصباح»** · From the wellspring to the lamp

Before you switch on a lamp tonight, energy will have made one journey. It is found, it is moved, it is stored, and then it is used. Ten teams follow that journey today. Each team owns one moment of it, and each moment has a problem that is hard for ordinary computers at full scale.

1. **المنبع · Source.** Find the energy and decide where to harvest it: read the rock under the ground, place the turbines in the wind.
2. **الطريق · Move.** Get it across the country: watch the grid, route the ships, decide which power stations run.
3. **المخزن · Store.** Keep it for later, and keep carbon out of the air: batteries, hydrogen, capture materials. This is chemistry.
4. **المصباح · Use.** Spend it well: make fresh water, charge and discharge a fleet of cars.

Your notebook is one chapter. The cell below lights your stop and lists all ten chapters so you can see who comes before and after you. When you pitch, start by saying where in the journey you are.

One rule for the whole challenge: **be honest about what is real.** As of November 2026 nobody has shown a quantum computer beating the best classical method on any of these ten problems. Your job is to explain the problem, show the idea on a small example, and say clearly what the small example does and does not prove.

In [ ]:
from qff.day import show_day
show_day(stop=2, challenge=5)

## 1 · This stop — هذه المحطة

**The problem.** A power system has to meet demand every hour, and the operator decides which power stations to switch on. Switching a unit on costs money, and a unit that is on must run. Too little capacity and the lights go out; too much and money is wasted. This is *unit commitment*: for each unit and each hour, on or off. Four units means 2⁴ = 16 plans for one hour; 26 units over 24 hours means 2⁶²⁴ plans.

**Who faces it.** A national grid operator, or a utility that owns a fleet of generating units, solves this every day for the next day.

<div style="border:1px solid rgba(208,38,112,0.35);border-left:6px solid #D02670;background:rgba(208,38,112,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#D02670;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🌍 Real world · في الواقع</span>

**What is real today.** WEF/Aramco *Quantum for Energy and Utilities* (April 2026), Case 13 (p. 27): IonQ and Oak Ridge National Laboratory, under the US DOE GRID-Q effort, ran a hybrid formulation for 26 generators over 24 hours on a 36-qubit trapped-ion system, using a layered variational method. The classical incumbent is mixed-integer programming in commercial solvers, which handles real systems with ramp limits, minimum up-times and reserves every day. (The paper's 76% survey figure belongs to "large-scale optimization of high-voltage grid flows", not to unit commitment.)

</div>

**Why a quantum computer might help.** Grover's algorithm finds a marked item among N in about √N steps instead of N, and wrapping it in a loop that keeps lowering a cost threshold turns "search" into "optimise". What is still missing: the quadratic speed-up only pays off on error-corrected hardware, and the circuit must compute the cost inside itself.

| Paper's method | Our teaching route |
|---|---|
| Layered variational method on a 36-qubit trapped-ion system, 26 generators over 24 hours (Case 13) | Grover adaptive search with Qiskit's `GroverOptimizer` on 4 generators for one hour. It builds directly on the Grover lecture and has a ready Qiskit class. This is a teaching choice, not the paper's method. |

## 2 · Made small — المسألة مصغَّرة

One peak hour, four generating units, demand **D = 5** (units of 100 MW). Unit *i* has capacity cᵢ, a start cost sᵢ, and a binary switch uᵢ (1 = on).

| Unit | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| Capacity cᵢ | 3 | 2 | 2 | 1 |
| Start cost sᵢ | 4 | 3 | 2 | 2 |

We minimise the start cost plus a penalty λ = 2 for every squared unit of mismatch with demand:

**cost(u) = Σ sᵢ uᵢ + λ (Σ cᵢ uᵢ − D)²**

Expanded, this is the QUBO `24 u0u1 + 24 u0u2 + 12 u0u3 + 16 u1u2 + 8 u1u3 + 8 u2u3 − 38 u0 − 29 u1 − 30 u2 − 16 u3 + 50`. The demand penalty is written by hand so you can see it. (The numbers are invented for teaching.)

Qubit budget: **4 decision qubits** (one per unit) **+ 7 value qubits** (the circuit has to hold the cost inside itself) = **11 qubits**. (Qiskit adds one more flag qubit inside the oracle, so the circuit it runs shows 12; we count 4 + 7.)

In [ ]:
QUBITS = 11                      # [KEEP] the test runner reads this
DECISION_QUBITS = 4
VALUE_QUBITS = 7
from qiskit import QuantumCircuit
from qiskit_optimization import QuadraticProgram
from qff.bits import index_to_x

def plan(x):
    """Unit switches as a string in unit order: [1, 0, 1, 0] -> '1010' (u0 first)."""
    return "".join(str(int(v)) for v in x)

CAPACITY = [3, 2, 2, 1]
START_COST = [4, 3, 2, 2]
LAMBDA = 2

def unit_commitment_qp(demand=5):
    """cost(u) = sum s_i u_i + LAMBDA * (sum c_i u_i - demand)^2, expanded by hand into linear and quadratic terms."""
    qp = QuadraticProgram("unit_commitment")
    for i in range(4):
        qp.binary_var(f"u{i}")
    linear = {f"u{i}": START_COST[i] + LAMBDA * (CAPACITY[i] ** 2 - 2 * demand * CAPACITY[i]) for i in range(4)}
    quadratic = {(f"u{i}", f"u{j}"): 2 * LAMBDA * CAPACITY[i] * CAPACITY[j] for i in range(4) for j in range(i + 1, 4)}
    qp.minimize(constant=LAMBDA * demand ** 2, linear=linear, quadratic=quadratic)
    return qp

DEMAND = 5
qp = unit_commitment_qp(DEMAND)
print(qp.prettyprint())

In [ ]:
# Classical baseline: always runs first.
import time
t0 = time.perf_counter()
classical_x, classical_value, table = qff.classical.brute_force(qp)
classical_seconds = time.perf_counter() - t0
print(f"brute force over {2 ** DECISION_QUBITS} states: best u = {plan(classical_x)}, cost {classical_value:g}")
print("cheapest five:", [(plan(x), v) for x, v, _ in table[:5]], "| worst:", (plan(table[-1][0]), table[-1][1]))

# A second baseline, "merit order": switch units on from cheapest start cost per MW until demand is met.
order = sorted(range(4), key=lambda i: START_COST[i] / CAPACITY[i])
merit_x, supplied = [0, 0, 0, 0], 0
for i in order:
    if supplied >= DEMAND:
        break
    merit_x[i], supplied = 1, supplied + CAPACITY[i]
merit_value = qp.objective.evaluate(merit_x)
print(f"merit order {order}: u = {plan(merit_x)}, supplies {supplied}, cost {merit_value:g}")

Merit order happens to find the optimum on this tiny instance. It is a rule of thumb, not a guarantee: it ignores how overshoot and start costs trade off, and it fails on harder instances. The point of the baseline is that **16 states are checked in microseconds**.

## 3 · The algorithm — الخوارزمية

### 3.0 · The map of this section

<div style="border:1px solid rgba(0,125,121,0.35);border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">💡 Idea · فكرة</span>

**Grover adaptive search in one sentence.** Give every plan an amplitude, mark the plans that are cheaper than the best one found so far by flipping the sign of their amplitude, let a second trick (inversion about the average) turn those sign flips into *larger* amplitudes for the marked plans, measure, and repeat with a lower bar until nothing cheaper exists.

</div>

There is **no knob-tuning** here (unlike QAOA or VQE). The circuit is fixed; what is cleverly arranged is how amplitudes move. We build it from two things you already know: lists of numbers (vectors) and probability. Every step has a **numbered equation** and, right after it, the **code cell that implements it**.

| Step | What happens | Equation | Where in the code | Your task |
|---|---|---|---|---|
| 3.1 | The cost of each plan, and a state with equal amplitude on all 16 plans | (1), (2) | `cost`, `by_hand` | **T1** |
| 3.2 | The oracle: flip the sign of every plan cheaper than the bar | (3) | `DiagonalGate` | |
| 3.3 | Inversion about the average, and one full Grover step | (4), (5) | `grover_operator` | |
| 3.4 | A rotation picture: how many steps are best? | (6), (7) | `p_marked_curve` | **T2** |
| 3.5 | The adaptive loop, and Qiskit's `GroverOptimizer` | (7) | `solve_gas` | |
| 3.6 | The value register: why the circuit needs 7 extra qubits | (8) | `reliability` | **T3** |

Words in **bold** are new ideas; each is explained where it first appears, and the Arabic term sits in a small table next to it.

In [ ]:
qff.plots.plot_flow(
    [("Pick a bar y", "cost of a random plan", "source"), ("Oracle", "mark plans cheaper than y", "move"), ("Grover steps", "random number", "store"),
     ("Measure", "one plan", "use"), ("Cheaper?", "then lower y", "source")],
    loop=(4, 1, "repeat; stop after several failures in a row"), title="Grover adaptive search");

**Symbol table** (keep it open while you read; the right column is where the symbol lives in the code).

| Symbol | Meaning | In the code |
|---|---|---|
| $x=(u_0,\dots,u_3)$ | a plan: unit $i$ on when $u_i=1$ | `plan(x)`, `index_to_x(k, 4)` |
| $\text{cost}(x),\ y$ | cost of a plan (equation 1); the bar (threshold) | `cost[k]`, `threshold` |
| $N,\ M$ | number of plans (16), number of marked plans | `len(cost)`, `len(marked)` |
| $\lvert s\rangle,\ a_x$ | equal mixture, amplitude of plan $x$ | `by_hand`, `Statevector(...).data` |
| $O,\ D,\ G=DO$ | oracle (sign flip), inversion about the average, one Grover step | `DiagonalGate`, `grover_operator` |
| $\theta,\ k$ | rotation angle per half step, number of Grover steps | `np.arcsin(np.sqrt(M / N))`, `steps` |
| $P(\text{marked})$ | probability of reading a marked plan | `p_marked_curve` |
| $v$ | width of the value register (qubits) | `num_value_qubits`, `bits_needed` |

<div style="border:1px solid rgba(15,98,254,0.35);border-left:6px solid #0F62FE;background:rgba(15,98,254,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#0F62FE;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">∑ Math · رياضيات</span>&nbsp; **The line to remember**

**The one line to remember.** After $k$ Grover steps $P(\text{marked})=\sin^2\bigl((2k+1)\theta\bigr)$ with $\sin\theta=\sqrt{M/N}$ (equation 6): each step *rotates* the state by $2\theta$, so the best $k$ is about $\frac\pi4\sqrt{N/M}$ and too many steps make things worse.

</div>

### 3.1 · The cost of every plan, and a state to search in

A **plan** is a list of four on/off switches $u=(u_0,u_1,u_2,u_3)$, one per unit, so there are $2^4=16$ plans. The cost of a plan is the one from section 2: the start costs of the units that are on, plus a penalty $\lambda=2$ for every squared unit of mismatch with the demand $D=5$:

$$\text{cost}(u)=\sum_i s_i\,u_i+\lambda\Bigl(\sum_i c_i\,u_i-D\Bigr)^2 \tag{1}$$

For example $u=(1,0,1,0)$: units 0 and 2 are on, start cost $4+2=6$, supply $3+2=5=D$, so no penalty: cost $6$. For $u=(0,0,0,0)$: nothing on, supply 0, penalty $2\cdot(0-5)^2=50$: cost $50$. Task T1 fills this table for all 16 plans.

**The state.** Four qubits hold a list of $2^4=16$ numbers called **amplitudes** $a_x$, one per plan; $|a_x|^2$ is the **probability** of reading out plan $x$ when you **measure**, and the probabilities add up to 1. The **Hadamard gate** $H$ puts one qubit into an equal mixture of $|0\rangle$ and $|1\rangle$; one $H$ on each of the four qubits (written $H^{\otimes4}$) gives every plan the same amplitude:

$$|s\rangle=H^{\otimes4}|0000\rangle=\frac1{\sqrt{16}}\sum_x|x\rangle,\qquad a_x=\frac14\ \text{for every plan},\qquad \sum_x a_x^2=16\cdot\frac1{16}=1 \tag{2}$$

($|x\rangle$ is just a *name tag* for plan $x$: a column of 16 numbers with a single 1 at position $x$.) This is pure guessing: each plan has probability $\frac1{16}$. Position $k$ of the list is the plan whose switch $u_i$ is bit $i$ of $k$ (so $k=5=0101_2$ is $u_0=1,u_2=1$, the plan `1010`).

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**Amplitudes can be negative.** Probabilities cannot, but amplitudes can: only their *squares* are probabilities. A minus sign is invisible to a measurement of that state, but it changes what later gates do. The whole trick of Grover's algorithm is built on a minus sign.

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Unit commitment | التزام الوحدات (جدولة تشغيل المولدات) | I | Measurement | القياس | G |
| Generating unit | وحدة توليد | I | Hadamard gate | بوابة هادامارد | G |
| Plan | الخطة | I | Cost function | دالة التكلفة | I |
| Amplitude | السعة | G | Superposition | التراكب | G |
| Probability | الاحتمال | I |  |  |  |

In [ ]:
from qiskit.circuit.library import DiagonalGate, grover_operator
from qiskit.quantum_info import Statevector

### 🧰 Toolbox: how the calls you will need are written · صندوق الأدوات

No internet? No problem. The cell under this table runs each call once on a **small toy that has nothing to do with the task**, so you can see the exact spelling, the order of the arguments, and what comes back. Run it, read the printed lines, and come back to it whenever a hint says "a method of ...". *What* to build, and with *which numbers*, is your job.

| To do | How it is written | What you get |
|---|---|---|
| a circuit with $n$ qubits | `qc = QuantumCircuit(n)` | all qubits start in $\lvert0\rangle$ |
| Hadamard on every qubit | `qc.h(range(n))` | one call puts the gate on each listed qubit |
| a list of $\pm1$ as a gate | `DiagonalGate([1, 1, 1, -1])` | multiplies the amplitude of position $k$ by the $k$-th number |
| put a gate on chosen qubits | `qc.append(gate, range(n))` | the gate is added at the end of `qc` |
| one full Grover step | `G = grover_operator(oracle_circuit)` | a **circuit**: the sign flip you give it, followed by the reflection about the average |
| put a circuit after another | `qc.compose(G, inplace=True)` | `G` is added at the end of `qc` |
| probabilities of all positions | `Statevector(qc).probabilities()` | a numpy array; **position $k$ means qubit $i$ equals bit $i$ of $k$** (so $k=2$ is qubit 1 set, qubit 0 clear) |

In [ ]:
# Toolbox for 05 and 09, on a 2-qubit toy with one marked position (not the notebook's cost table).
from qiskit import QuantumCircuit
from qiskit.circuit.library import DiagonalGate, grover_operator
from qiskit.quantum_info import Statevector

n_toy = 2
sign_flip = QuantumCircuit(n_toy)
sign_flip.append(DiagonalGate([1, 1, 1, -1]), range(n_toy))      # flips the sign of position 3 only
toy_step = grover_operator(sign_flip)                            # one full Grover step, as a circuit
toy = QuantumCircuit(n_toy)
toy.h(range(n_toy))                                              # the equal start state
print("before any step:", np.round(Statevector(toy).probabilities(), 3))
toy.compose(toy_step, inplace=True)                              # one Grover step after it
probs_toy = Statevector(toy).probabilities()
print("after one step :", np.round(probs_toy, 3), "<- position", int(np.argmax(probs_toy)), "now has almost all the probability")
assert abs(probs_toy[3] - 1) < 1e-9
print("OK: toolbox ran.")

### T1 · Warm-up: the cost table · التمرين ١

**What you are about to build, and why.** Grover's oracle (next) will mark *every plan cheaper than a bar*. To understand it, you need the cost of all 16 plans in front of you. In T1 you fill the list `cost` (one integer per position $k=0..15$), and the list of plans whose cost is below 8 (the plans the oracle would mark at the bar $y=8$).

**The formula you implement:** equation (1) for every plan. `qp.objective.evaluate(x)` computes it for you given the switches $x$; your job is to produce the 16 switch lists from the positions $k$ (bit $i$ of $k$ is unit $i$) and collect the results.

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

You need to visit the 16 positions k = 0, ..., 15. For each one you need the four switches as a list, with bit i of k as unit i: the helper `index_to_x(k, 4)`, imported in section 2, does exactly that (it returns the list $[x_0,x_1,x_2,x_3]$ for position `k`). Then ask the problem for the cost of that list.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

A list comprehension over the 16 positions builds `cost`: turn each position into its switch list, evaluate the objective of the problem on it, and round to an integer. For the marked plans, go through the positions again and keep the plan string of every position whose cost is below the bar.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
cost = ______              # 16 integers: the cost of plan k, for k = 0..15
marked_thr8 = ______       # the plan strings whose cost is below 8
```
Test yourself: `cost[5]` (the plan `1010`) is 6, and four plans are cheaper than 8.

</details>

In [ ]:
# TODO (team) T1: fill the 16-row cost table and find the plans cheaper than 8
# Goal: `cost` holds one integer per position k = 0..15 (bit i of k is unit i); `marked_thr8` holds the plans (strings like "1100") with cost below 8.
cost = []
marked_thr8 = []
raise NotImplementedError("T1: fill the cost table and the marked plans")

In [ ]:
# Self-check
assert len(cost) == 16, "T1: the table needs one cost for each of the 16 plans"
assert cost == [50, 12, 21, 7, 20, 6, 7, 17, 34, 8, 13, 11, 12, 10, 7, 29], "T1: a cost is off; check bit i of k is unit i"
assert sorted(marked_thr8) == sorted(["1100", "1010", "0110", "0111"]), "T1: four plans are cheaper than 8: 1100, 1010, 0110, 0111"
print("marked for threshold 8:", marked_thr8)

In [ ]:
# The 16 costs as a picture: the plans the oracle will mark (cost below 8) are the low teal bars.
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(range(16), cost, color=["#007D79" if c < 8 else "#8D8D8D" for c in cost])
ax.axhline(8, color="#FA4D56", ls="--", lw=1, label="threshold 8: plans below it are marked")
ax.set_xticks(range(16)); ax.set_xticklabels([plan(index_to_x(k, DECISION_QUBITS)) for k in range(16)], rotation=90, fontsize=7)
ax.set_xlabel("plan (one bit per unit)"); ax.set_ylabel("cost"); ax.legend(fontsize=8)
plt.tight_layout()
assert sum(c < 8 for c in cost) == len(marked_thr8) == 4, "the bars below the dashed line are exactly the marked plans"
print("OK: four teal bars sit below the threshold, and they are the four marked plans.")

In [ ]:
# Equation (1): cost(u) = sum_i s_i u_i + lambda (sum_i c_i u_i - D)^2, recomputed for all 16 plans with the notebook's own constants.
DEMAND = 5
cost_by_formula = [sum(START_COST[i] * u[i] for i in range(4)) + LAMBDA * (sum(CAPACITY[i] * u[i] for i in range(4)) - DEMAND) ** 2
                   for u in (index_to_x(k, DECISION_QUBITS) for k in range(16))]
assert cost_by_formula == cost, "equation (1) disagrees with the cost table of T1"
print("OK: equation (1) reproduces the cost table for all 16 plans.")

### 3.2 · The oracle: flip the sign of every cheap plan

Pick a **bar** $y$. A plan is **marked** if its cost is below $y$. The **oracle** $O$ is a rule that flips the sign of the amplitude of every marked plan and leaves the others alone:

$$O\,|x\rangle=\begin{cases}-|x\rangle&\text{if }\text{cost}(x)<y\ \ (\text{marked})\\+|x\rangle&\text{otherwise}\end{cases} \tag{3}$$

As a matrix, $O$ is *diagonal*: a $16\times16$ table with $\pm1$ on its diagonal and zeros elsewhere. In Qiskit that is a `DiagonalGate` built from the list of $\pm1$ values. A sign flip changes **no probability** ($(-a)^2=a^2$), so right after the oracle a measurement still sees every plan equally likely. The information "which plans are cheap" is now stored in the *signs* where a measurement cannot see it, and the next step turns it into amplitude.

**Careful.** This oracle looks at the cost table directly, which a real algorithm cannot do (the real oracle computes costs *inside* the circuit, section 3.6). Here it is for intuition only.

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Oracle | العرّاف (oracle) | I | Diagonal matrix | مصفوفة قطرية | I |
| Marked plan | الخطة الموسومة | I | Threshold (the bar) | العتبة | I |
| Phase / sign flip | قلب الإشارة (الطور) | I |  |  |  |

### 3.3 · Inversion about the average

The second ingredient is a **reflection about the average**. Let $\bar a$ be the average of the 16 amplitudes. The step replaces every amplitude by

$$a_x\ \longmapsto\ 2\bar a-a_x \tag{4}$$

In words: *mirror each amplitude around the average.* An amplitude that is *far below* the average (a marked plan, after its sign flip) lands *far above* it; an amplitude close to the average barely moves. (As a matrix this is $D=2|s\rangle\langle s|-I$: twice the projection onto the equal state, minus the identity.) One full **Grover step** is "oracle, then inversion about the average":

$$G=D\,O \tag{5}$$

**A worked example you can check on paper: 4 plans, 1 of them marked.** Amplitudes start at $\frac12$ each. The oracle flips the marked one: $(\frac12,\frac12,\frac12,-\frac12)$. The average is $\bar a=\frac14(\frac12+\frac12+\frac12-\frac12)=\frac14$. Inversion (4): the three unmarked become $2\cdot\frac14-\frac12=0$ and the marked one becomes $2\cdot\frac14+\frac12=1$. Result $(0,0,0,1)$: **one step finds the marked plan with certainty.** (Check: $0^2+0^2+0^2+1^2=1$ ✓.) The cell below does this arithmetic with numpy, then the same for our 16 plans.

<div style="border:1px solid rgba(0,125,121,0.35);border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">💡 Idea · فكرة</span>

**Why this amplifies.** The sign flip pulls the marked amplitudes *below* the average; the mirror pushes them *above* it by about the same distance, while the average itself moves only a little. Net effect: marked plans grow, unmarked plans shrink. Repeat, and the probability of a marked plan keeps growing, up to a point (next section).

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Grover's algorithm | خوارزمية جروفر | G | Average (mean) | المتوسط | I |
| Amplitude amplification | تضخيم السعة | I | Interference | التداخل | G |
| Reflection / inversion about the mean | الانعكاس (القلب) حول المتوسط | I | Quantum speedup (quadratic) | تسريع تربيعي | U |

In [ ]:
# Equations (2)-(5) with plain numpy: sign flip, then inversion about the average. No quantum library.
def grover_by_hand(cost, bar, steps=3):
    """Amplitudes of the 16 plans after each action. Returns (dict name -> amplitudes, list of P(marked) after 0..steps steps, marked positions)."""
    c = np.array(cost)
    marked = np.flatnonzero(c < bar)
    a = np.full(len(c), 1 / np.sqrt(len(c)))                          # equation (2): equal amplitudes
    stages, curve = {"start": a.copy()}, [float((a[marked] ** 2).sum())]
    for step in range(1, steps + 1):
        a = a.copy()
        a[marked] *= -1                                               # equation (3): the oracle flips the sign of marked plans
        stages[f"step {step}: oracle"] = a.copy()
        a = 2 * a.mean() - a                                          # equation (4): inversion about the average
        stages[f"step {step}: invert"] = a.copy()
        curve.append(float((a[marked] ** 2).sum()))
    return stages, curve, marked


# the 4-plan example from the text: one plan marked
tiny = np.full(4, 0.5)
tiny[3] *= -1
print("4 plans, 1 marked: after the oracle", tiny, "-> after inversion about the average", 2 * tiny.mean() - tiny)
assert np.allclose(2 * tiny.mean() - tiny, [0, 0, 0, 1])

stages8, by_hand_thr8, marked8 = grover_by_hand(cost, 8)
print("16 plans, bar 8 (4 marked): P(marked) after 0, 1, 2, 3 steps =", np.round(by_hand_thr8, 3))
sel = {k: v for k, v in stages8.items() if k in ("start", "step 1: oracle", "step 1: invert", "step 2: oracle", "step 2: invert")}
qff.plots.plot_amplitude_stages(sel, highlight=marked8, labels=("not marked", "marked"),
                                suptitle="bar 8: four plans are marked (magenta). The oracle flips their sign, the inversion lifts them")

<div style="border:2px dashed #007D79;border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🧪 Try it · جرّب</span>

Change the bar `8` to `7` in the last cell (so only one plan, `1010`, is marked) and rerun it with `steps=3`. After the oracle the marked amplitude is slightly negative; after the inversion it is much bigger than the others. **Predict** which step gives the highest P(marked) before you look at the printed curve.

</div>

### 3.4 · Why it is a rotation, and how many steps are best

Let $N=16$ be the number of plans and $M$ the number of marked ones. Group the amplitudes into two classes, marked and not marked. Initially the state is a mix with $P(\text{marked})=M/N$; write this as $\sin^2\theta=M/N$. Each Grover step $G=DO$ **rotates** the state by the angle $2\theta$ in the plane of the two classes, towards "all marked". After $k$ steps the state sits at angle $(2k+1)\theta$, so

$$P(\text{marked after }k\text{ steps})=\sin^2\bigl((2k+1)\theta\bigr),\qquad \sin\theta=\sqrt{M/N} \tag{6}$$

Two things follow. First, the probability **rises, peaks and falls again**, because rotating past "all marked" turns the state back. Second, the number of steps that gets closest to the peak is about

$$k^*\approx\frac\pi{4\theta}-\frac12\ \approx\ \frac\pi4\sqrt{\frac NM}\quad(\text{for small }M/N) \tag{7}$$

For $M=4$, $N=16$: $\sin\theta=\frac12$, so $\theta=\frac\pi6$; step 1 reaches $\sin^2\frac\pi2=1$, and step 2 gives $\sin^2\frac{5\pi}6=\frac14$ again (the state rotated *past* the target). For $M=1$: $\sin\theta=\frac14$, $\theta\approx0.253$; $k=3$ gives $\sin^2(7\theta)=0.961$ and is the best. **The best number of steps depends on $M$, which the search does not know.** That is why Grover *adaptive* search draws the number of steps at random (section 3.5). Compared with a classical search that checks about $N/2$ plans on average, about $\sqrt N$ steps is the **quadratic speed-up**: $4$ steps instead of $16$ looks small here, but $10^6$ steps instead of $10^{12}$ does not.

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Rotation | الدوران | I | Number of steps | عدد الخطوات | I |
| Quadratic speed-up | تسريع تربيعي | U |  |  |  |

In [ ]:
# Equations (6) and (7) as a picture: each Grover step turns the state by 2*theta.
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
qff.plots.plot_grover_geometry(np.arcsin(np.sqrt(4 / 16)), steps=2, ax=axes[0])
axes[0].set_title("4 of 16 marked: theta = 30 deg; step 1 hits, step 2 overshoots", fontsize=8)
qff.plots.plot_grover_geometry(np.arcsin(np.sqrt(1 / 16)), steps=3, ax=axes[1])
axes[1].set_title("1 of 16 marked: theta = 14.5 deg; best at step 3", fontsize=8)
plt.tight_layout()

### T2 · Core step: the oracle and the Grover step in Qiskit · التمرين ٢

**What you are about to build, and why.** The numpy cell above showed what a Grover step does to the amplitudes. In T2 you build the same thing with Qiskit gates, as a function `p_marked_curve(threshold)`: it builds the oracle (equation 3) as a `DiagonalGate`, the full step $G=DO$ (equation 5) with Qiskit's `grover_operator`, starts from $H^{\otimes4}$ (equation 2), applies $G$ zero, one, two, … times, and records $P(\text{marked})$ after each. You will then see that Qiskit's circuit gives the same numbers as your pencil-and-paper arithmetic and as equation (6).

**The formulas you implement:** (2), (3), (5), and $P(\text{marked})=\sum_{k\in\text{marked}}|a_k|^2$ (read from `Statevector(qc).probabilities()`).

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

Break it into pieces. (1) Which positions are marked? (2) The oracle is a circuit on 4 qubits that contains one `DiagonalGate`, which takes a list of 16 numbers (which value for a marked state, which for the others?). (3) Qiskit turns an oracle circuit into the full Grover step with a ready-made function. (4) To apply the step k times, start from the equal mixture and repeat.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

Inside `p_marked_curve`: find the marked positions; build the oracle circuit with one `DiagonalGate`; get the step with `grover_operator`; then for each number of steps from 0 to `max_steps` build a circuit (Hadamard on every qubit, then the step applied that many times with `compose`), take the probabilities of its `Statevector`, and add up the marked ones. Return the list of probabilities and the step.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
def p_marked_curve(threshold, max_steps=4):
    marked = ______            # the marked positions
    oracle = ______            # a 4-qubit circuit with the diagonal gate of equation (3)
    G = ______                 # the full Grover step
    curve = []
    for steps in range(max_steps + 1):
        qc = ______            # equal mixture, then the step applied `steps` times
        curve.append(______)   # probability of the marked positions, as a float
    return curve, G
```
Self-check targets: with the bar at 8 the curve starts at 0.25, reaches 1.0 after one step and falls back to 0.25 after two.

</details>

In [ ]:
# TODO (team) T2: build the phase oracle and the Grover operator, find the best number of steps
def p_marked_curve(threshold, max_steps=4):
    # Goal: p_marked_curve(threshold, max_steps=4) returns (the list of P(marked) after 0..max_steps Grover steps, the Grover step G),
    # where "marked" means cost < threshold, using the oracle (equation 3) and the step (equation 5) of section 3.
    raise NotImplementedError("T2: build the phase oracle and the Grover operator")

p_marked_thr8, G8 = p_marked_curve(8)
p_marked_thr7, G7 = p_marked_curve(7)
best_steps_thr7 = int(np.argmax(p_marked_thr7))

In [ ]:
# Self-check
assert abs(p_marked_thr8[0] - 0.25) < 1e-6, "T2: before any step, 4 of 16 plans are marked, so P = 0.25"
assert abs(p_marked_thr8[1] - 1.0) < 1e-6, "T2: with 4 of 16 marked, one Grover step should give P = 1"
assert abs(p_marked_thr8[2] - 0.25) < 1e-6, "T2: a second step undoes it (P = 0.25 again)"
assert best_steps_thr7 == 3, "T2: with one marked plan the best number of steps is 3"
print("threshold 8:", np.round(p_marked_thr8, 3))
print("threshold 7:", np.round(p_marked_thr7, 3), "-> best steps:", best_steps_thr7)

**Check against pencil-and-paper arithmetic and equation (6).** The cell below compares your Qiskit curves with the numpy arithmetic of 3.3 and with $\sin^2((2k+1)\theta)$, for both bars, and then plots them.

In [ ]:
# Check: Qiskit's curves = the numpy arithmetic = equation (6).
for bar, qiskit_curve in ((8, p_marked_thr8), (7, p_marked_thr7)):
    _, hand_curve, marked_k = grover_by_hand(cost, bar, steps=4)
    theta = np.arcsin(np.sqrt(len(marked_k) / 16))
    formula = [np.sin((2 * k + 1) * theta) ** 2 for k in range(5)]
    assert np.allclose(qiskit_curve, hand_curve) and np.allclose(qiskit_curve, formula), f"bar {bar}: the three curves should agree"
    print(f"bar {bar}: {len(marked_k)} plan(s) marked, theta = {np.degrees(theta):.1f} deg | Qiskit {np.round(qiskit_curve, 3)} | formula (6) {np.round(formula, 3)}")
print("OK: Qiskit, pencil and paper, and equation (6) all give the same numbers.")
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, curve, label, M in ((axes[0], p_marked_thr8, "bar 8: 4 plans marked", 4), (axes[1], p_marked_thr7, "bar 7: 1 plan marked", 1)):
    fine = np.linspace(0, 4, 200)
    ax.plot(fine, np.sin((2 * fine + 1) * np.arcsin(np.sqrt(M / 16))) ** 2, color="#C6C6C6", label="equation (6), any k")
    ax.plot(range(len(curve)), curve, "o", color="#0F62FE", label="Qiskit")
    ax.set_title(label); ax.set_xlabel("Grover steps"); ax.set_xticks(range(len(curve))); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("P(marked plan)")
plt.tight_layout(); plt.show()
print("The best number of steps depends on how many plans are marked, and the loop does not know that in advance.")
print("That is why Grover adaptive search draws the number of steps at random.")

In [ ]:
# The Grover step as a circuit (the oracle you built in T2, then the reflection about the equal mixture), for the bar 8.
try:
    display(G8.decompose().draw("mpl", fold=40))
except Exception:                                      # drawing with matplotlib needs pylatexenc; fall back to text
    print(G8.decompose().draw("text", fold=120))

### 3.5 · The adaptive loop, and Qiskit's `GroverOptimizer`

Grover's algorithm needs an oracle that marks the answers we want. For optimisation we do not know the answers in advance, so **Grover adaptive search** builds the oracle from a bar $y$: *mark every plan that costs less than $y$*. The loop, in words:

```
pick a random plan; let y = its cost
repeat:
    mark every plan with cost < y            (the oracle, computed *inside* the circuit)
    run Grover with a random number of steps  (we do not know how many plans are marked)
    measure one plan
    if its cost < y:  y = that cost           (the bar goes down)
    else:             count a failure
stop after several failures in a row
```

When the bar is low, few plans are marked ($M$ small) and many steps are best; when the bar is high, many plans are marked and few steps are best (equation 7). Because $M$ is unknown, the number of steps is drawn at random each round; this idea of searching with an unknown number of marked items goes back to Boyer, Brassard, Høyer and Tapp (1998), and its use for finding a minimum to Dürr and Høyer (1996). Qiskit's `GroverOptimizer` follows the same pattern for polynomial binary problems (it cites Gilliam, Woerner and Gonciulea). Each round that finds a cheaper plan lowers the bar, so the search finishes when *no plan is cheaper*: the best plan.

**The expected cost of the idea.** A classical program checks the 16 plans one by one: 16 lookups. The loop above needs *several* rounds, each of a few Grover steps, each step an oracle that computes the cost inside the circuit. On 16 plans this is **slower** than brute force; the speed-up is about $\sqrt N$ per round, and it only pays when $N$ is astronomically large and the hardware is error-corrected.

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Grover adaptive search | البحث التكيفي بخوارزمية جروفر | I | Optimization | الأمثَلة (التحسين) | I |
| Threshold (the bar) | العتبة | I | Value register | سجل القيمة | I |
| Random number of steps | عدد عشوائي من الخطوات | I |  |  |  |

**The real thing: `GroverOptimizer`.** Qiskit's `GroverOptimizer` runs the loop above. It builds the oracle *without* peeking: it computes the cost of every plan inside the circuit, in a **value register** of binary digits, subtracts the bar, and marks the plans whose result is negative. That register is why the circuit has 11 qubits, not 4. The next section explains how it works and how it fails.

Three things to know:
- `num_value_qubits` is the width of the value register. It must hold the signed range of (cost − bar). **Too few and the answers are silently wrong.**
- `num_iterations` is the number of *consecutive rounds without improvement* before it stops. It is not the number of Grover steps.
- Costs must be integers. Both seeds matter: `algorithm_globals.random_seed` and the sampler's `seed`.

We wrap it in one helper, `solve_gas`, and a tiny recording sampler so we can look at the circuits it really runs.

In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.utils import algorithm_globals
from qiskit_optimization.algorithms import GroverOptimizer

class RecordingSampler(StatevectorSampler):
    """A StatevectorSampler that keeps the circuits it is asked to run (so we can inspect them)."""
    def __init__(self, **kw):
        super().__init__(**kw)
        self.circuits = []
    def run(self, pubs, **kw):
        self.circuits.extend(p if isinstance(p, QuantumCircuit) else p[0] for p in pubs)
        return super().run(pubs, **kw)

def solve_gas(problem, value_qubits, seed, iterations=15, sampler=None):
    """One Grover adaptive search run. Sets BOTH seeds, then calls GroverOptimizer.solve."""
    algorithm_globals.random_seed = seed
    sampler = sampler or StatevectorSampler(seed=seed)
    return GroverOptimizer(num_value_qubits=value_qubits, num_iterations=iterations, sampler=sampler).solve(problem)

recorder = RecordingSampler(seed=SEED)
t0 = time.perf_counter()
gas_result = solve_gas(qp, VALUE_QUBITS, SEED, sampler=recorder)
gas_seconds = time.perf_counter() - t0
gas_circuits = len(gas_result.operation_counts)
print(f"x = {plan([int(v) for v in gas_result.x])}, cost = {gas_result.fval:g}, status = {gas_result.status}")
print(f"decision qubits {gas_result.n_input_qubits}, value qubits {gas_result.n_output_qubits}, circuits run {gas_circuits}, {gas_seconds:.1f} s")
circuit = max(recorder.circuits, key=lambda c: c.depth())        # the deepest circuit it ran: used by the IBM preview below
print(f"deepest circuit it ran: {circuit.num_qubits} qubits, depth {circuit.decompose(reps=2).depth()} after two decompositions")

### 3.6 · The value register: why 7 extra qubits, and how it fails

The oracle must decide "is $\text{cost}(x)<y$?" for all plans *at once*, inside the circuit. It does this with binary arithmetic on extra qubits, the **value register**: it computes the difference $\text{cost}(x)-y$ in the register and looks at the **sign bit** (the top qubit): 1 means negative, i.e. marked. Negative numbers are stored in **two's complement**: with $v$ bits the register can hold the whole numbers

$$-2^{\,v-1},\ \dots,\ -1,\ 0,\ 1,\ \dots,\ 2^{\,v-1}-1 \tag{8}$$

and anything outside this range **wraps around**. A tiny example, $v=3$: the range is $-4..3$, the bit patterns $100,101,110,111$ mean $-4,-3,-2,-1$, so the value $+5$ (which does not fit) is stored as $101$ and read as $-3$. A *positive* difference has turned into a *negative* one: an expensive plan looks cheap, and the oracle **marks the wrong plan**.

For our problem costs run from 6 to 50, and the bar $y$ is always one of the costs, so $\text{cost}-y$ can be as low as $6-50=-44$ and as high as $50-6=+44$. Equation (8) says 6 bits cover only $-32..31$ (**too few**); 7 bits cover $-64..63$ (enough). This is a simplified picture of what Qiskit's oracle does (its arithmetic circuit is more elaborate), but the overflow rule is the same. The cell below checks it and shows the failure: with 6 bits and a bar of 12, the *worst* plan (`0000`, cost 50, difference $+38$) wraps to $-26$ and is marked.

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**Silent failure.** With too few value qubits the routine still reports success and returns a wrong plan, sometimes even `0000` at cost 50. Nothing warns you. **Always check a quantum optimiser's answer against something you can compute classically.**

</div>

<sub>Arabic terms for this step. Tier G: in Quantabic's glossary. Tier I: proposed, to be checked against quantabic.com.</sub>

| English | العربية | Tier | English | العربية | Tier |
|---|---|---|---|---|---|
| Value register | سجل القيمة | I | Overflow | فيض (تجاوز المدى) | I |
| Two's complement | المتمم الثنائي | I | Qubit | كيوبت | G |
| Sign bit | بت الإشارة | I | Binary number | عدد ثنائي | I |

In [ ]:
# Equation (8): what a v-bit two's complement register can hold, and what overflow does to the oracle.
def wrap(value, bits):
    """The number a `bits`-bit two's complement register shows when asked to hold `value`."""
    return (value + 2 ** (bits - 1)) % 2 ** bits - 2 ** (bits - 1)


print("3-bit example: the value 5 is read as", wrap(5, 3), "| the value -4 is read as", wrap(-4, 3), "| the value 3 is read as", wrap(3, 3))
lo_needed, hi_needed = min(cost) - max(cost), max(cost) - min(cost)               # range of (cost - bar) when the bar is one of the costs
print(f"costs run from {min(cost)} to {max(cost)}, so cost - bar ranges over [{lo_needed}, {hi_needed}]")
print(f"{'value qubits':>13}{'register holds':>20}{'enough?':>10}")
for v in (5, 6, 7, 8):
    lo, hi = -2 ** (v - 1), 2 ** (v - 1) - 1
    print(f"{v:>13}{f'{lo} .. {hi}':>20}{str(lo <= lo_needed and hi_needed <= hi):>10}")


def bits_needed(costs):
    """Smallest register width that holds every cost - bar, with the bar one of the costs (equation 8)."""
    lo, hi = min(costs) - max(costs), max(costs) - min(costs)
    v = 1
    while not (-2 ** (v - 1) <= lo and hi <= 2 ** (v - 1) - 1):
        v += 1
    return v


assert bits_needed(cost) == 7
bar = 12
for v in (6, 7):
    wrongly_marked = [plan(index_to_x(k, 4)) for k in range(16) if cost[k] >= bar and wrap(cost[k] - bar, v) < 0]
    print(f"bar {bar}, {v} value qubits: plans marked although they are NOT cheaper than the bar: {wrongly_marked or 'none'}")
fig, ax = plt.subplots(figsize=(7, 2.6))
for i, v in enumerate((5, 6, 7, 8)):
    ax.barh(i, 2 ** v - 1, left=-2 ** (v - 1), color="#0F62FE" if (-2 ** (v - 1) <= lo_needed and hi_needed <= 2 ** (v - 1) - 1) else "#FA4D56")
ax.axvspan(lo_needed, hi_needed, color="#24A148", alpha=0.25, label="range needed: cost - bar")
ax.set_yticks(range(4)); ax.set_yticklabels([f"{v} value qubits" for v in (5, 6, 7, 8)]); ax.set_xlabel("whole numbers the register can hold")
ax.legend(fontsize=8, loc="lower right"); plt.tight_layout()

<div style="border:2px dashed #007D79;border-left:6px solid #007D79;background:rgba(0,125,121,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#007D79;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🧪 Try it · جرّب</span>

Change `bar = 12` to `bar = 7`, `20` or `30` and rerun the last cell. For which bars does the 6-qubit register mislabel a plan? Why does the mislabelling happen for the *worst* plans (high cost) and not the cheap ones?

</div>

### Reliability, and the value-register lesson

One run can be lucky. We repeat for seeds 1 to 10 with a 7-qubit value register, then for seeds 1 to 3 with only **6** (those runs are slow, so we run fewer). The costs run from 6 to 50, and the oracle works with (cost − threshold), whose signed range needs 7 bits. Six bits overflow.

In [ ]:
# Takes about 1 to 1.5 minutes in total (13 solves; the 6-qubit solves are slow because they wander for 50 to 150 circuits). Both seeds are set inside solve_gas.
def reliability(value_qubits, seeds=range(1, 11)):
    """Run GAS once per seed; return the list of (seed, x as string, cost, status)."""
    rows = []
    for s in seeds:
        r = solve_gas(qp, value_qubits, s)
        rows.append((s, plan([int(v) for v in r.x]), r.fval, str(r.status).split(".")[-1]))
    return rows

gas_rows = {}
for nv, seeds in ((7, range(1, 11)), (6, range(1, 4))):
    gas_rows[nv] = reliability(nv, seeds)
    hits = sum(1 for _, xs, v, _ in gas_rows[nv] if xs == plan(classical_x) and v == classical_value)
    print(f"{nv} value qubits: optimum found in {hits} of {len(gas_rows[nv])} seeds")
    print("   ", [(s, xs, int(v)) for s, xs, v, _ in gas_rows[nv]])
gas_hits_7bits = sum(1 for _, xs, v, _ in gas_rows[7] if xs == plan(classical_x) and v == classical_value)
gas_hits_6bits = sum(1 for _, xs, v, _ in gas_rows[6] if xs == plan(classical_x) and v == classical_value)
print("status reported with 6 value qubits:", sorted({st for *_, st in gas_rows[6]}), "<- it still says SUCCESS")

With 6 value qubits the routine reports success and returns a wrong plan, sometimes even `0000` at cost 50. Nothing warns you. **Always check a quantum optimiser's answer against something you can compute.**

### T3 · Stretch: the width of the value register, and a new demand · التمرين ٣

**What you are about to build, and why.** The failure of a narrow value register is the most important lesson of this notebook, and T3 lets you see it yourself. You run Grover adaptive search with 6, 7 and 8 value qubits on the same instance (same seed), and explain in one sentence *why* the width matters, using equation (8) and the range of $\text{cost}-y$. Then you change the demand to 6 (a new instance), solve it classically first, and solve it with Grover adaptive search using enough value qubits. You may use the helper `bits_needed` from the cell above to decide how many.

**The formulas you use:** the register holds $-2^{v-1}..2^{v-1}-1$ (equation 8); the oracle needs $[\min\text{cost}-\max\text{cost},\ \max\text{cost}-\min\text{cost}]$.

**Stuck? Open the hints one at a time, and try again after each. · تلميحات**

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 1 · a nudge · تلميح ١</summary>

Look at the table in 3.6: which widths hold the range -44 to 44 and which do not? You do not need new maths: you need three calls of the helper `solve_gas(problem, value_qubits, seed)`, which returns an object with the chosen plan and its cost. Remember that `plan(...)` turns a list of switches into a string.

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 2 · the structure · تلميح ٢</summary>

Structure: for 6, 7 and 8 value qubits, solve with seed 2 and store the plan string and the cost; write one sentence using equation (8); build the instance for demand 6 with `unit_commitment_qp`; solve it classically first with `brute_force`; then solve it with `solve_gas` using a register wide enough for the new range (`bits_needed(costs)` helps: it takes a *list of costs*, so for the new instance take the costs of all plans from the table that `brute_force` returns, and remember that a bigger demand can need more bits than the 7 of the main example).

</details>

<details style="border:1px solid rgba(128,128,128,0.45);border-radius:8px;padding:4px 12px;margin:6px 0">
<summary style="cursor:pointer;font-weight:600">Hint 3 · nearly there · تلميح ٣</summary>

```python
wide = {}
for nv in (6, 7, 8):
    ______                     # solve with nv value qubits; wide[nv] = (plan string, cost)
explanation = "..."            # one sentence, using the range of cost - bar
qp6 = ______                   # the instance for demand 6
x6, v6, _ = ______             # classical answer first
r6 = ______                    # Grover adaptive search with enough value qubits
```

</details>

In [ ]:
# TODO (team) T3: try 6, 7 and 8 value qubits, explain the failures, then change the demand to 6
# Goal: wide[nv] = (plan string, cost) from solve_gas on the demand-5 instance with nv = 6, 7, 8 value qubits (seed 2);
# explanation = one sentence on why the value register needs 7 bits here; then build the demand-6 instance qp6 (the problem with demand 6),
# solve it classically first (x6 = best plan, v6 = its cost), and then with Grover adaptive search (r6 = the result object of solve_gas), and compare.
# The Self-check reads wide, explanation, qp6, x6, v6 and r6.
wide = {}
explanation = ""
qp6, x6, v6, r6 = None, None, None, None
raise NotImplementedError("T3: try 6, 7 and 8 value qubits, then change the demand")

In [ ]:
# Self-check
assert set(wide) == {6, 7, 8}, "T3: run 6, 7 and 8 value qubits"
assert len(explanation.strip()) > 20, "T3: write one sentence about the value register"
assert v6 == qp6.objective.evaluate(x6), "T3: the classical answer for demand 6 should be the best plan"
assert r6.fval == v6, "T3: with enough value qubits GAS should match the classical optimum for demand 6"

### Optional: how this would go to a real IBM Quantum computer
This is a sketch and has not been checked against the current IBM documentation. Nothing here runs in this notebook.

```python
# Not run here. Needs: pip install qiskit-ibm-runtime, and an IBM Quantum account saved on your own machine.
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# service = QiskitRuntimeService()                      # reads credentials saved earlier; never paste a token in a notebook
# backend = service.least_busy(operational=True, simulator=False)
# isa = generate_preset_pass_manager(optimization_level=3, backend=backend).run(circuit)
# job = Sampler(mode=backend).run([isa], shots=4096)
```

The next cell does the offline part: it transpiles your circuit for a *pretend* device on this computer and reports its size. Nothing is sent to IBM Quantum.

In [ ]:
preview = qff.quantum.device_preview(circuit)      # `circuit` = the main circuit of this notebook; runs locally
print(f"depth {preview['depth']}, two-qubit gates {preview['two_qubit_gates']}")

<div style="border:1px solid rgba(208,38,112,0.35);border-left:6px solid #D02670;background:rgba(208,38,112,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#D02670;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">🌍 Real world · في الواقع</span>

The preview above is the deepest circuit `GroverOptimizer` ran (4 decision qubits, 7 value qubits and the oracle qubit). A circuit with thousands of two-qubit gates would **not** survive today's noisy devices: every gate adds a little error and the answer would drown in noise. Grover's quadratic speed-up needs error-corrected hardware, and that is why our claim in section 4 stays modest.

</div>

**Glossary — المصطلحات.** Tier G = Quantabic headword, U = used consistently on Quantabic, I = proposed here and not yet checked against quantabic.com. Where Quantabic has a term, we use it.

| English | العربية | Tier |
|---|---|---|
| Qubit | كيوبت | G |
| Superposition | التراكب | G |
| Entanglement | التشابك | G |
| Measurement | القياس | G |
| Amplitude | السعة | G |
| Phase | الطور | U |
| Quantum circuit | الدائرة الكمومية | U |
| Quantum gate | البوابة الكمومية | U |
| Hadamard gate | بوابة هادامارد | G |
| Quantum algorithm | خوارزمية كمومية | U |
| Quantum simulator | المحاكي الكمومي | I |
| Classical bit | البت الكلاسيكي | I |
| Quantum speedup (quadratic / exponential) | تسريع تربيعي / أسي | U |
| Grover's algorithm | خوارزمية جروفر | G |
| oracle | العرّاف (oracle) | I |
| amplitude amplification | تضخيم السعة | I |
| optimization | التحسين (الأمثَلة) | I |
| threshold | العتبة | I |
| value register | سجل القيمة | I |
| two's complement | المتمم الثنائي | I |

## 4 · Compare — المقارنة

In [ ]:
# Results table: classical baselines first.
rows = [
    ("brute force", plan(classical_x), classical_value, f"{2 ** DECISION_QUBITS} states", f"{classical_seconds * 1000:.2f} ms"),
    ("merit order", plan(merit_x), merit_value, "4 steps", "-"),
    (f"Grover adaptive, {VALUE_QUBITS} value qubits", plan([int(v) for v in gas_result.x]), gas_result.fval,
     f"{gas_circuits} circuits ({QUBITS} qubits)", f"{gas_seconds:.1f} s"),
]
print(f"{'method':<40}{'plan':<8}{'cost':<7}{'states / circuits':<26}{'time'}")
for m, plan_str, v, work, sec in rows:
    print(f"{m:<40}{plan_str:<8}{v:<7g}{work:<26}{sec}")
print(f"\nReliability: {gas_hits_7bits}/10 seeds with 7 value qubits, {gas_hits_6bits}/3 seeds with 6.")

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.bar(["6 value qubits", "7 value qubits"], [gas_hits_6bits / 3, gas_hits_7bits / 10], color=["#FA4D56", "#0F62FE"])
ax.set_ylim(0, 1); ax.set_ylabel("share of seeds that found the optimum"); ax.set_title("Width of the value register")
plt.tight_layout(); plt.show()

**What this toy shows.** Grover's amplification can be wrapped in a loop that keeps lowering a threshold until nothing cheaper exists. On 16 schedules it finds the optimum every time with a wide enough value register, and it fails silently when the register is too narrow.

<div style="border:1px solid rgba(138,63,252,0.35);border-left:6px solid #8A3FFC;background:rgba(138,63,252,0.1);border-radius:8px;padding:6px 16px 8px 16px;margin:10px 0">

<span style="background:#8A3FFC;color:#fff;border-radius:12px;padding:2px 10px;font-size:0.85em;font-weight:600;white-space:nowrap">⚠️ Watch out · انتبه</span>

**What this toy does not show.** The quantum circuit computes the cost inside itself, which took 7 extra qubits and deep arithmetic for a 4-unit problem; the loop above needed 16 or more circuit runs where brute force needed 16 lookups. Grover's speed-up is quadratic at best (about √N instead of N checks) and needs error-corrected hardware at useful sizes. Real unit commitment has ramp limits, minimum up-times, reserves and 24 or more hours, and classical mixed-integer solvers handle it daily. Nothing here is a demonstration of quantum advantage.

</div>

**Paper versus our route.** WEF/Aramco Case 13 (p. 27): IonQ and Oak Ridge National Laboratory, under the US DOE GRID-Q effort, ran a hybrid formulation for 26 generators over 24 hours on a 36-qubit trapped-ion system, using a layered variational method on hardware. We chose Grover adaptive search instead because it builds directly on the Grover lecture and has a ready Qiskit class. This notebook is our teaching route, not the paper's method, and it ran on a simulator.

In [ ]:
# Acceptance: the numbers this notebook promises
assert QUBITS == 11 and DECISION_QUBITS == 4
assert classical_x == [1, 0, 1, 0] and classical_value == 6
assert abs(p_marked_thr8[1] - 1.0) < 1e-6 and abs(p_marked_thr8[2] - 0.25) < 1e-6
assert int(np.argmax(p_marked_thr7)) == 3 and abs(p_marked_thr7[3] - 0.961) < 0.01
assert list(map(int, gas_result.x)) == [1, 0, 1, 0] and gas_result.fval == 6
assert gas_hits_7bits >= 9 and gas_hits_6bits < 3, f"hits: 7 bits {gas_hits_7bits}, 6 bits {gas_hits_6bits}"
print("acceptance passed:", dict(gas_hits_7bits=gas_hits_7bits, gas_hits_6bits=gas_hits_6bits, circuits=gas_circuits))

## 5 · Build the explainer — ابنوا المنتَج
You now build one product, **in either Arabic or English**, that teaches a visitor your use case and your algorithm together. It is 50% of your score, and the product should be something that does not take more than 5 minutes.

**Your audience:** a first-year university student who has never seen a qubit. They have three minutes.

**Your product must answer four questions, in this order:**
1. What is the problem, and who has it? Which power stations should run tonight, at least cost, while still meeting demand?
2. Why is it hard when it gets big? 4 units give 16 on/off plans; 26 units over 24 hours give 2⁶²⁴ plans
3. What does the quantum algorithm do differently? mark every plan cheaper than your best so far, amplify the marked ones, measure, repeat with a lower bar
4. How real is it today? (Use your section 4 results and the honesty statement.)

**Pick any format:** it can be anything **except a presentation (.ppt file)**. It can be an interactive demo, an interactive website, or a YouTube video (or a small app, a short deck, a game, or an interactive notebook)—as long as it does not take more than 5 minutes.

**Three ideas that fit this stop** (take one, change it, or ignore them):
1. A "limbo bar" animation where the cost threshold drops each round
2. A control-room game: switch units on and off against the clock, then race the circuit
3. A physical card-trick version of amplitude amplification (flip the marked cards, reflect, repeat)

**Use from this notebook:** the P(marked)-versus-steps plot, the 6-versus-7 value-qubit failure table

**How it is scored (Product, 50%)**
| Criterion | Weight | What judges look for | Where it comes from |
|---|---|---|---|
| Accuracy | 17 | Every claim is true; the classical baseline is shown; nothing is oversold | Sections 1, 2, 4 |
| Conceptual insight, in Arabic | 17 | A newcomer understands *why* the algorithm works, in clear Arabic | Sections 3, 4 |
| Creativity | 11 | A fresh way to show it: metaphor, interaction, story | This section |
| Execution | 5 | It works, it is finished, it is readable | This section |

Qiskit integration and code is a separate 10%: keep your completed notebook tidy and show one cell of it.

**Arabic terms.** Use the glossary cell at the end of section 3. Keep gate letters, kets and symbols in Latin (H, |0⟩, θ). Use Quantabic's terms. If a term is not in the glossary, check quantabic.com, and if it is not there either, give the English in brackets the first time.

**The words are yours.** Mentors can check your physics; they will not write your Arabic.

**Time plan for the 2 h 20 build window:** 20 min agree the story and format · 70 min build · 20 min test on someone outside the team · 30 min rehearse the pitch.

## 6 · Pitch — العرض
You have **5 minutes** for your pitch and 3 minutes for questions from the judges. Presentation is 40% of your score.

| Minutes | What | Who |
|---|---|---|
| 0–0.5 | Where we are in the journey: stop, use case, one sentence | 1 speaker |
| 0.5–1.5 | The problem and why it is hard at scale | |
| 1.5–3 | The algorithm, with your toy example and one notebook cell | |
| 3–4 | Live demo of your Arabic product | |
| 4–5 | How real is it: your result against the classical baseline | |
| 5-8 | Questions | all |

**Questions you should expect**
1. Did the quantum method beat the classical one? (Answer honestly: It found the cheapest plan 10 times out of 10 on 16 options, using 11 simulated qubits; a loop over 16 options is faster.)
2. What happens when the problem is 100 times bigger?
3. Why this algorithm and not another?
4. Why does a second Grover step make things worse?
5. What happens with too few value qubits?

**Checklist before you walk up**
- [ ] The notebook runs top to bottom and the self-checks pass
- [ ] Every number on screen comes from your notebook or from a named source
- [ ] You say what the toy does not show
- [ ] You finish the talk in 5 minutes on a timer